# Generation of a simple DFN with PorePy and Transfer to OGS

This is work in progress

In [66]:
import numpy as np
import porepy as pp
import ogstools as ogs
import os as os

# Setting up the domain and generating a random set of circular fractures

In [46]:
mins = np.array([0.,0.,0.])
maxs = np.array([10.,10.,10.])

In [47]:
bounding_box = {'xmin': mins[0], 'xmax': maxs[0], 'ymin': mins[1], 'ymax': maxs[1], 'zmin': mins[2], 'zmax': maxs[2]}
domain = pp.Domain(bounding_box=bounding_box)
domain

pp.Domain(bounding_box={'xmin': 0.0, 'xmax': 10.0, 'ymin': 0.0, 'ymax': 10.0, 'zmin': 0.0, 'zmax': 10.0})

In [48]:
nfracs = 8
r_range = np.array([3,9])
f_i = np.array([])
for i in range(nfracs):
    center = np.random.rand(3) * (maxs - mins) + mins
    major_axis = np.random.rand() * (r_range[1] - r_range[0]) + r_range[0]
    minor_axis = major_axis.copy() #circular
    major_axis_angle = 0. #for circular
    strike_angle = np.random.rand() * np.pi - np.pi/2
    dip_angle = np.random.rand() * np.pi - np.pi/2
    f_i = np.append(f_i,pp.create_elliptic_fracture(center, major_axis, minor_axis, major_axis_angle, strike_angle, dip_angle))

In [49]:
network = pp.create_fracture_network(fractures=f_i,domain=domain)
network

Three-dimensional fracture network with 8 plane fractures.
The domain is a cuboid with bounding box: {'xmin': 0.0, 'xmax': 10.0, 'ymin': 0.0, 'ymax': 10.0, 'zmin': 0.0, 'zmax': 10.0}.

## Meshing ... 

In [50]:
mesh_args = {'cell_size_boundary': 1.0, 'cell_size_fracture': 0.5, 'cell_size_min': 0.2}
mdg = pp.create_mdg("simplex", mesh_args, network)

In [57]:
#Removal of 3D not needed really
#mdg2d = mdg.copy()
#for sd in mdg2d.subdomains():
#    if sd.dim == 3:
#        mdg2d.remove_subdomain(sd)
#mdg2d

In [52]:
#pp.plot_grid(mdg2d, figsize=(12,12), plot_2d=False)

## Export to VTU and import in OGS. Setting up Material IDs

In [58]:
pp.Exporter(mdg, 'mixed_dimensional_grid').write_vtu()

In [59]:
DFN_2D = ogs.Mesh('mixed_dimensional_grid_constant_2.vtu')
DFN_2D

Mesh (0x7f0f2b6cdf60)
  N Cells:    10152
  N Points:   5969
  X Bounds:   0.000e+00, 1.000e+01
  Y Bounds:   0.000e+00, 1.000e+01
  Z Bounds:   0.000e+00, 1.000e+01
  N Arrays:   6

In [60]:
DFN_2D['MaterialIDs'] = DFN_2D['subdomain_id'] - DFN_2D['subdomain_id'].min()

In [61]:
fig = DFN_2D.plot('MaterialIDs',show_edges=True)

Widget(value='<iframe src="http://localhost:41839/index.html?ui=P_0x7f0f2b701310_5&reconnect=auto" class="pyvi…

## Generating boundaries for OGS

In [67]:
cmd = '~/build/release2/bin/ExtractBoundary -i mixed_dimensional_grid_constant_2.vtu -o boundaries.vtu'
os.system(cmd)

[2025-03-29 14:29:25.508] [ogs] [info] Mesh read: 5969 nodes, 10152 elements.
[2025-03-29 14:29:25.509] [ogs] [info] 6 property vectors copied, 0 vectors skipped.
[2025-03-29 14:29:25.509] [ogs] [info] Created surface mesh: 1712 nodes, 1712 elements.


0

In [71]:
tol = 1e-3
cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o xmax.vtu --x-max %.3f' %(maxs[0]-tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o xmin.vtu --x-min %.3f' %(mins[0]+tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o ymax.vtu --y-max %.3f' %(maxs[1]-tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o ymin.vtu --y-min %.3f' %(mins[1]+tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o zmax.vtu --z-max %.3f' %(maxs[2]-tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o zmin.vtu --z-min %.3f' %(mins[2]+tol)
os.system(cmd)

[2025-03-29 14:37:43.316] [ogs] [info] Mesh read: 1712 nodes, 1712 elements.
[2025-03-29 14:37:43.317] [ogs] [info] Bounding box of "boundaries" is
x = [0.000000,10.000000]
y = [0.000000,10.000000]
z = [0.000000,10.000000]
[2025-03-29 14:37:43.317] [ogs] [info] 1664 elements found.
[2025-03-29 14:37:43.317] [ogs] [info] Removing total 1664 elements...
[2025-03-29 14:37:43.317] [ogs] [info] 48 elements remain in mesh.
[2025-03-29 14:37:43.317] [ogs] [info] Removing total 1659 nodes...
[2025-03-29 14:37:43.348] [ogs] [info] Mesh read: 1712 nodes, 1712 elements.
[2025-03-29 14:37:43.348] [ogs] [info] Bounding box of "boundaries" is
x = [0.000000,10.000000]
y = [0.000000,10.000000]
z = [0.000000,10.000000]
[2025-03-29 14:37:43.348] [ogs] [info] 1668 elements found.
[2025-03-29 14:37:43.348] [ogs] [info] Removing total 1668 elements...
[2025-03-29 14:37:43.348] [ogs] [info] 44 elements remain in mesh.
[2025-03-29 14:37:43.348] [ogs] [info] Removing total 1664 nodes...
[2025-03-29 14:37:43.3

0